# Practical Implementation: Extraction (PCA) & Construction

This notebook demonstrates how to process a customer churn dataset through two distinct feature engineering lenses:
1. **Feature Extraction via PCA**: Algorithmically reducing dimensionality to visualize cluster separation.
2. **Feature Construction**: Manually leveraging domain knowledge to build a specific predictive metric.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder

sns.set_theme(style="whitegrid")

### 1. Generate Synthetic Telco Data
We will generate a synthetic dataset mimicking the Telco Customer Churn data to ensure this notebook runs independently.

In [ ]:
np.random.seed(42)
n_samples = 2000

# Basic Features
tenure = np.random.randint(1, 72, n_samples)
monthly_charges = np.random.uniform(20, 120, n_samples)
# TotalCharges is roughly tenure * monthly_charges + some noise
total_charges = tenure * monthly_charges + np.random.normal(0, 100, n_samples)
total_charges = np.where(total_charges < 0, 0, total_charges) # No negative charges

# Categorical Features
contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2])
internet = np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples)

# Target Variable: Churn (Higher probability if Month-to-month, high monthly charges, low tenure)
churn_prob = np.where(contract == 'Month-to-month', 0.4, 0.1) + (monthly_charges/120)*0.2 - (tenure/72)*0.3
churn = np.random.binomial(1, np.clip(churn_prob, 0, 1))

# Compile DataFrame
df = pd.DataFrame({
    'customerID': [f'CUST_{i:04d}' for i in range(n_samples)],
    'Tenure': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'Contract': contract,
    'InternetService': internet,
    'Churn': churn
})

display(df.head())

### 2. Feature Extraction via PCA
**Objective:** Transform the data into Principal Components to reduce noise and visualize the data in 2D space.

**Steps:** Drop IDs -> Encode Categories -> Scale Features -> Apply PCA.

In [ ]:
# Preprocessing
df_ml = df.copy()

# 1. Drop Non-Predictive Features
df_ml.drop('customerID', axis=1, inplace=True)

# 2. Categorical Encoding using LabelEncoder
le = LabelEncoder()
categorical_features = ['Contract', 'InternetService']
for col in categorical_features:
    df_ml[col] = le.fit_transform(df_ml[col])

# Separate features and target
X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

# 3. Scaling (Crucial for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. PCA Extraction
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Original feature shape: {X.shape}")
print(f"PCA feature shape: {X_pca.shape}")
print(f"Variance explained by 2 components: {np.sum(pca.explained_variance_ratio_)*100:.2f}%")

### 3. Visual Diagnostic: PCA Scatter Plot
Let's map the newly extracted features (Principal Components) to an X and Y axis to see if the algorithm successfully grouped churned customers together.

In [ ]:
pca_df = pd.DataFrame(data=X_pca, columns=['Principal Component 1', 'Principal Component 2'])
pca_df['Churn'] = y.replace({0: 'Retained', 1: 'Churned'})

plt.figure(figsize=(10, 7))
sns.scatterplot(x='Principal Component 1', y='Principal Component 2', hue='Churn', 
                data=pca_df, palette=['teal', 'coral'], alpha=0.6, s=50)
plt.title('2D PCA Projection of Telco Customer Data')
plt.show()

print("Insight: PCA compresses our multi-dimensional data into a 2D map. Notice how the 'Churned' customers (orange) tend to cluster toward specific regions of the principal component space.")

### 4. Feature Construction
While PCA is entirely mathematical, **Feature Construction** relies on domain knowledge.

Here, we explicitly combine `TotalCharges`, `MonthlyCharges`, and `Tenure` to create a metric representing the "average intensity" of a customer's spending relative to their loyalty.

In [ ]:
# Construct the new feature
df['avg_charges_per_service'] = df['TotalCharges'] / (df['MonthlyCharges'] + df['Tenure'])

# Cleaning: Handle division by zero or NaN issues
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

display(df[['Tenure', 'MonthlyCharges', 'TotalCharges', 'avg_charges_per_service', 'Churn']].head())

# Let's visualize if this constructed feature has predictive power
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['Churn'].replace({0: 'Retained', 1: 'Churned'}), y=df['avg_charges_per_service'], palette='Set2')
plt.title("Constructed Feature: Avg Charges Per Service by Churn Status")
plt.ylabel("Avg Charges Per Service (Index)")
plt.show()

print("Insight: By constructing this single domain-specific feature, we might capture a behavioral nuance (like billing irregularities or high usage intensity) that pure mathematical extraction (PCA) might obscure.")